In [1]:
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [2]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [3]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [4]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [5]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [6]:
rec = answers[0]

In [7]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [8]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key points from the ground truth: late enrollment/participation is allowed, and certificate eligibility depends on submitting the project while submissions are still open. This is semantically equivalent.', score='good')

In [9]:
calc_price(usage)

{'input_cost': 0.00024000000000000003,
 'output_cost': 0.000252,
 'total_cost': 0.000492}

In [10]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [11]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the core meaning of the ground truth: late enrollment/participation is allowed, but certificate eligibility depends on submitting the project before submissions close. This is semantically equivalent and correct.', score='good')

In [12]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [13]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/765 [00:00<?, ?it/s]

In [14]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [15]:
df_eval = pd.DataFrame(evaluations)

In [16]:
calc_total_price(usages)

0.56723925

In [17]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 722/765 = 94.38%


In [18]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
2,"Do latecomers get to take part in the course, ...",74eb249bbf,bad,The ground truth says latecomers can still tak...
6,"If I signed up for the course, should I still ...",977bf7786c,bad,The AI answer does not convey the ground truth...
31,What do I actually need to pass in this course...,9f689c185f,bad,The AI answer preserves the key point that pas...
32,Are the homework assignments mandatory for cer...,9f689c185f,bad,The AI answer fails to provide the ground trut...
41,Do you know when the course starts again after...,bd31146b0e,bad,The ground truth gives a specific restart time...


In [19]:
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)